# Phân tích dữ liệu thử nghiệm Pilot (RBL-4 - Tuần 7)

Tài liệu này thực hiện các yêu cầu của mục **7.3 Phân tích pilot** và **7.4 Quyết định sau pilot** theo cẩm nang RBL-4:
1. Tính toán metric mô tả trên tập Pilot (descriptive statistics).
2. Trực quan hóa phân phối bằng biểu đồ.
3. Xác nhận lựa chọn kiểm định thống kê và đưa ra quyết định tiếp tục dự án.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập cấu hình hiển thị biểu đồ
%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Dữ liệu thực nghiệm Pilot (6 hàm mẫu)
Dưới đây là bảng số liệu chi tiết thu được sau khi chạy sinh test case từ AI (*GPTTest) và đo đạc qua JaCoCo và PITest.

In [ ]:
data = {
    "class_name": ["BF", "COUNT_NUMS", "FILE_NAME_CHECK", "FIND_ZERO", "ORDER_BY_POINTS", "SEARCH"],
    "branch_coverage": [100.00, 100.00, 84.62, 87.50, 85.71, 100.00],
    "mutation_score": [92.86, 88.24, 88.46, 78.95, 90.48, 92.31]
}

df = pd.DataFrame(data)
df

## 2. Thống kê mô tả (Descriptive Statistics)
Tính toán các chỉ số thống kê cơ bản như Trung vị (Median), Trung bình (Mean), Min, Max, và Độ lệch chuẩn (Std).

In [ ]:
desc_stats = df.describe().loc[['mean', '50%', 'min', 'max', 'std']]
desc_stats.rename(index={'50%': 'median'}, inplace=True)
desc_stats

## 3. Trực quan hóa phân phối (Distribution Plots)
Vẽ biểu đồ Boxplot để quan sát phân phối dữ liệu của Branch Coverage và Mutation Score trên tập Pilot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot cho Branch Coverage
sns.boxplot(y="branch_coverage", data=df, ax=axes[0], color="skyblue", width=0.4)
sns.stripplot(y="branch_coverage", data=df, ax=axes[0], color="black", size=6, jitter=0.15)
axes[0].set_title("Phân phối Branch Coverage (AI Test)")
axes[0].set_ylabel("Tỷ lệ bao phủ (%)")
axes[0].set_ylim(0, 105)

# Boxplot cho Mutation Score
sns.boxplot(y="mutation_score", data=df, ax=axes[1], color="salmon", width=0.4)
sns.stripplot(y="mutation_score", data=df, ax=axes[1], color="black", size=6, jitter=0.15)
axes[1].set_title("Phân phối Mutation Score (AI Test)")
axes[1].set_ylabel("Điểm đột biến (%)")
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

## 4. Xác nhận kiểm định thống kê (Statistical Test Selection)
Dựa trên phân tích phân phối mẫu thử nghiệm Pilot:
1. **Tính chất dữ liệu:** Paired Data (Dữ liệu cặp) - mỗi lớp SUT đều được đo đạc song song bởi 2 phương pháp (AI và EvoSuite).
2. **Phân phối:** Cả Branch Coverage và Mutation Score đều giới hạn trong khoảng [0, 100]% và bị lệch nặng về phía cận trên (skewed towards 100%), không tuân theo phân phối chuẩn (normal distribution).
3. **Kết luận:** Lựa chọn kiểm định **Wilcoxon Signed-Rank Test** (kiểm định phi tham số cho dữ liệu cặp) đã ghi nhận trong Proposal là hoàn toàn chính xác và phù hợp để sử dụng cho thực nghiệm chính thức ở Tuần 8.

## 5. Quyết định sau Pilot (7.4)
* **Đánh giá Pipeline:** Pipeline chạy đúng, biên dịch thành công. Lỗi thiếu `tools.jar` đã được khắc phục bằng cách cấu hình lại POM. Lỗi test case AI bị fail trên code gốc đã được giải quyết tự động bằng script `ignore_failing_tests.py` để đạt suite "Green" cho PITest.
* **Tỷ lệ hợp lệ (Valid Rate):** Đạt 100% (không có response nào bị rỗng hay lỗi API không sinh được test).
* **Quyết định:** **TIẾN HÀNH experiment chính thức Tuần 8 trên toàn bộ 63 hàm**.